# AHS-KT × ASSIST2009 全流程 Notebook

这本 Notebook 的目标不是简单复用现成 `npz`，而是**直接从你指定的数据目录** `/root/autodl-tmp/ahs-kt/data/assist2009` 出发，把：

1. 用户切分读取；
2. `assist2009_ahs.pkl` 序列加载；
3. 题目/概念映射；
4. 难度统计；
5. AHS 行为聚类与软原型；
6. `AHS-KT` 训练与评估；
7. `acc / auc / f1` 输出；

完整串起来。

## 为什么这里不直接套旧版 `assist2009_v5`

从第一性原理看，你这次明确给了数据目录 `/root/autodl-tmp/ahs-kt/data/assist2009`。
所以最短且最干净的路径，不是绕回另一套预生成产物，而是：

- 直接读取这个目录中的 `assist2009_ahs.pkl`；
- 直接读取这个目录中的 `train_valid.csv` / `test.csv`；
- 在 Notebook 里重建本次实验专用的 `.npz / metadata / config`；
- 再调用项目原生 `AHSKTModel + fit_and_evaluate` 跑通。

这样得到的是一份真正 **self-contained** 的 `assist2009 all-in-one` notebook。

In [1]:
from pathlib import Path
import json
import os
import pickle
import random
import sys
import time

import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path('/root/autodl-tmp/ahs-kt')
SRC_ROOT = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data' / 'assist2009'
ARTIFACT_DIR = PROJECT_ROOT / 'data' / 'assist2009_all_20260421'
TRAIN_NPZ_PATH = ARTIFACT_DIR / 'assist2009_all_20260421_train_ahskt.npz'
VALID_NPZ_PATH = ARTIFACT_DIR / 'assist2009_all_20260421_valid_ahskt.npz'
TEST_NPZ_PATH = ARTIFACT_DIR / 'assist2009_all_20260421_test_ahskt.npz'
METADATA_PATH = ARTIFACT_DIR / 'assist2009_all_20260421_metadata.json'
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'ahskt_assist2009_all_20260421.json'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'assist2009_all_20260421_run'

SEED = 20260421
VALID_FOLD = 0
SEQUENCE_LENGTH = 100
REMAINDER_MIN_LEN = 10
QUESTION_ALPHA = 5.0
CONCEPT_ALPHA = 20.0
NUM_BEHAVIOR_PROTOTYPES = 4
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
PATIENCE = 2
TASK_NAME = 'ahskt_assist2009_all_20260421'

assert PROJECT_ROOT.exists(), f'找不到项目目录: {PROJECT_ROOT}'
assert DATA_DIR.exists(), f'找不到数据目录: {DATA_DIR}'
assert (DATA_DIR / 'assist2009_ahs.pkl').exists(), '缺少 assist2009_ahs.pkl'
assert (DATA_DIR / 'train_valid.csv').exists(), '缺少 train_valid.csv'
assert (DATA_DIR / 'test.csv').exists(), '缺少 test.csv'
assert (DATA_DIR / 'keyid2idx.json').exists(), '缺少 keyid2idx.json'

os.chdir(PROJECT_ROOT)
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import tensorflow as tf
from ahskt.config import load_config
from ahskt.data.assist2009 import save_bundle_npz
from ahskt.data.dataset import SequenceBundle, load_bundle_from_config
from ahskt.models.ahs_kt import AHSKTModel
from ahskt.training.engine import fit_and_evaluate

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_DIR =', DATA_DIR)
print('TensorFlow version =', tf.__version__)
print('GPU devices =', tf.config.list_physical_devices('GPU'))
for gpu_device in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError:
        pass

PROJECT_ROOT = /root/autodl-tmp/ahs-kt
DATA_DIR = /root/autodl-tmp/ahs-kt/data/assist2009
TensorFlow version = 2.8.0
GPU devices = [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. 构建辅助函数

下面这些函数负责四件事：

- 规范化原始键值；
- 从 train split 统计平滑难度；
- 把变长用户序列切成固定长度 `SequenceBundle`；
- 在测试集上补算 `f1`。

In [2]:
def normalize_key(value):
    text = str(value)
    if '_' in text:
        return text
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except ValueError:
        pass
    return text


def compute_smoothed_maps(frame: pd.DataFrame, id_col: str, response_col: str, alpha: float):
    grouped = frame.groupby(id_col)[response_col].agg(['sum', 'count'])
    global_mean = float(frame[response_col].mean()) if len(frame) else 0.5
    default_bin = int(np.clip(int(global_mean * 100) + 1, 1, 101))
    posterior = (grouped['sum'] + alpha * global_mean) / (grouped['count'] + alpha)
    bins = np.clip((posterior * 100).astype(int) + 1, 1, 101).astype(int)
    if alpha > 0:
        confidence = grouped['count'] / (grouped['count'] + alpha)
    else:
        confidence = pd.Series(np.ones(len(grouped), dtype=np.float32), index=grouped.index)
    return (
        bins.to_dict(),
        posterior.astype(float).to_dict(),
        confidence.astype(float).to_dict(),
        default_bin,
        global_mean,
    )


def count_sequences(records, sequence_length, remainder_min_len):
    total = 0
    for payload in records.values():
        length = len(payload['question_ids'])
        total += length // sequence_length
        if length % sequence_length >= remainder_min_len:
            total += 1
    return total


def allocate_bundle(num_sequences, sequence_length, num_behavior_clusters):
    return {
        'question_ids': np.zeros((num_sequences, sequence_length), dtype=np.int32),
        'concept_ids': np.zeros((num_sequences, sequence_length), dtype=np.int32),
        'responses': np.zeros((num_sequences, sequence_length), dtype=np.int32),
        'question_difficulty': np.zeros((num_sequences, sequence_length), dtype=np.int32),
        'concept_difficulty': np.zeros((num_sequences, sequence_length), dtype=np.int32),
        'attempts': np.zeros((num_sequences, sequence_length), dtype=np.float32),
        'hints': np.zeros((num_sequences, sequence_length), dtype=np.float32),
        'speed': np.zeros((num_sequences, sequence_length), dtype=np.float32),
        'speed_relative_student': np.zeros((num_sequences, sequence_length), dtype=np.float32),
        'speed_relative_question': np.zeros((num_sequences, sequence_length), dtype=np.float32),
        'behavior_cluster': np.zeros((num_sequences, sequence_length), dtype=np.int32),
        'behavior_soft_membership': np.zeros((num_sequences, sequence_length, num_behavior_clusters), dtype=np.float32),
        'mask': np.zeros((num_sequences, sequence_length), dtype=np.int32),
        'question_easiness': np.zeros((num_sequences, sequence_length), dtype=np.float32),
        'concept_easiness': np.zeros((num_sequences, sequence_length), dtype=np.float32),
        'question_confidence': np.zeros((num_sequences, sequence_length), dtype=np.float32),
        'concept_confidence': np.zeros((num_sequences, sequence_length), dtype=np.float32),
    }


def build_sequence_bundle(
    records,
    sequence_length,
    remainder_min_len,
    q_diff_map,
    c_diff_map,
    default_q_diff,
    default_c_diff,
    q_ease_map,
    c_ease_map,
    q_conf_map,
    c_conf_map,
    default_q_ease,
    default_c_ease,
    default_confidence,
    num_behavior_clusters,
):
    num_sequences = count_sequences(records, sequence_length, remainder_min_len)
    payload = allocate_bundle(num_sequences, sequence_length, num_behavior_clusters)
    row = 0
    for uid in sorted(records):
        record = records[uid]
        length = len(record['question_ids'])
        if length < 2:
            continue
        enriched = {
            'question_ids': record['question_ids'],
            'concept_ids': record['concept_ids'],
            'responses': record['responses'],
            'question_difficulty': np.asarray([q_diff_map.get(int(x), default_q_diff) for x in record['question_ids']], dtype=np.int32),
            'concept_difficulty': np.asarray([c_diff_map.get(int(x), default_c_diff) for x in record['concept_ids']], dtype=np.int32),
            'attempts': record['attempts'],
            'hints': record['hints'],
            'speed': record['speed'],
            'speed_relative_student': record['speed_relative_student'],
            'speed_relative_question': record['speed_relative_question'],
            'behavior_cluster': record['behavior_cluster'],
            'behavior_soft_membership': record['behavior_soft_membership'],
            'question_easiness': np.asarray([q_ease_map.get(int(x), default_q_ease) for x in record['question_ids']], dtype=np.float32),
            'concept_easiness': np.asarray([c_ease_map.get(int(x), default_c_ease) for x in record['concept_ids']], dtype=np.float32),
            'question_confidence': np.asarray([q_conf_map.get(int(x), default_confidence) for x in record['question_ids']], dtype=np.float32),
            'concept_confidence': np.asarray([c_conf_map.get(int(x), default_confidence) for x in record['concept_ids']], dtype=np.float32),
        }
        full_chunks = length // sequence_length
        for chunk_index in range(full_chunks):
            begin = chunk_index * sequence_length
            end = begin + sequence_length
            for key, values in enriched.items():
                payload[key][row, ...] = values[begin:end]
            payload['mask'][row, :] = 1
            row += 1
        left = length % sequence_length
        if left >= remainder_min_len:
            begin = full_chunks * sequence_length
            end = length
            for key, values in enriched.items():
                payload[key][row, :left, ...] = values[begin:end]
            payload['mask'][row, :left] = 1
            row += 1
    return SequenceBundle(**payload)


def collect_targets_and_predictions(model, bundle, batch_size):
    dataset = bundle.to_tf_dataset(batch_size=batch_size, shuffle=False)
    all_targets = []
    all_predictions = []
    for batch in dataset:
        logits = model(batch, training=False)
        next_logits = logits[:, :-1]
        next_targets = tf.cast(batch['responses'][:, 1:], tf.float32)
        next_mask = tf.cast(batch['mask'][:, 1:], tf.float32)
        valid_logits = tf.boolean_mask(next_logits, next_mask > 0)
        valid_targets = tf.boolean_mask(next_targets, next_mask > 0)
        all_targets.append(valid_targets.numpy())
        all_predictions.append(tf.sigmoid(valid_logits).numpy())
    return np.concatenate(all_targets, axis=0), np.concatenate(all_predictions, axis=0)

## 2. 从 `assist2009` 目录直接重建 AHS-KT bundle

这一节会：

- 读取 `assist2009_ahs.pkl`；
- 使用 `train_valid.csv` 的 `fold != 0` 做 train，`fold == 0` 做 valid；
- 使用 `test.csv` 做 test；
- 直接从原始序列自建 question/concept 映射；
- 基于 train split 拟合平滑难度与 AHS 行为聚类；
- 导出本次 notebook 专用的 `.npz / metadata / config`。

In [3]:
with open(DATA_DIR / 'keyid2idx.json', 'r', encoding='utf-8') as f:
    keyid2idx = json.load(f)
with open(DATA_DIR / 'assist2009_ahs.pkl', 'rb') as f:
    raw_sequences = pickle.load(f)
train_valid_df = pd.read_csv(DATA_DIR / 'train_valid.csv', usecols=['fold', 'uid'])
test_df = pd.read_csv(DATA_DIR / 'test.csv', usecols=['uid'])

train_uids = set(train_valid_df.loc[train_valid_df['fold'] != VALID_FOLD, 'uid'].astype(int))
valid_uids = set(train_valid_df.loc[train_valid_df['fold'] == VALID_FOLD, 'uid'].astype(int))
test_uids = set(test_df['uid'].astype(int))

uid_index = {normalize_key(key): int(value) for key, value in keyid2idx['uid'].items()}
all_question_keys = sorted({normalize_key(value) for payload in raw_sequences.values() for value in payload['questions']})
all_concept_keys = sorted({normalize_key(value) for payload in raw_sequences.values() for value in payload['concepts']})
question_index = {key: idx + 1 for idx, key in enumerate(all_question_keys)}
concept_index = {key: idx + 1 for idx, key in enumerate(all_concept_keys)}

records_by_split = {'train': {}, 'valid': {}, 'test': {}}
missing_uid_keys = set()
for raw_uid, raw_payload in raw_sequences.items():
    uid_key = normalize_key(raw_uid)
    if uid_key not in uid_index:
        missing_uid_keys.add(uid_key)
        continue
    mapped_uid = int(uid_index[uid_key])
    if mapped_uid in train_uids:
        split = 'train'
    elif mapped_uid in valid_uids:
        split = 'valid'
    elif mapped_uid in test_uids:
        split = 'test'
    else:
        continue

    question_ids = [question_index[normalize_key(value)] for value in raw_payload['questions']]
    concept_ids = [concept_index[normalize_key(value)] for value in raw_payload['concepts']]
    record = {
        'question_ids': np.asarray(question_ids, dtype=np.int32),
        'concept_ids': np.asarray(concept_ids, dtype=np.int32),
        'responses': np.asarray(raw_payload['responses'], dtype=np.int32),
        'attempts_raw': np.asarray(raw_payload['attempts'], dtype=np.float32),
        'hints_raw': np.asarray(raw_payload['hints'], dtype=np.float32),
        'speed_raw': np.asarray(raw_payload['speeds'], dtype=np.float32),
    }
    records_by_split[split][mapped_uid] = record

assert not missing_uid_keys, f'存在缺失 uid 映射: {list(sorted(missing_uid_keys))[:5]}'

print('split users =', {name: len(records_by_split[name]) for name in ['train', 'valid', 'test']})
print('num_questions =', len(question_index))
print('num_concepts =', len(concept_index))

train_rows = []
for uid, record in records_by_split['train'].items():
    train_rows.append(pd.DataFrame({
        'uid': uid,
        'question_id': record['question_ids'],
        'concept_id': record['concept_ids'],
        'response': record['responses'],
        'attempt_raw': record['attempts_raw'],
        'hint_raw': record['hints_raw'],
        'speed_raw': record['speed_raw'],
    }))
train_frame = pd.concat(train_rows, ignore_index=True)
train_frame['attempt_log'] = np.log1p(np.clip(train_frame['attempt_raw'].astype(np.float64), a_min=0.0, a_max=None))
train_frame['hint_log'] = np.log1p(np.clip(train_frame['hint_raw'].astype(np.float64), a_min=0.0, a_max=None))
train_frame['speed_log'] = np.log1p(np.clip(train_frame['speed_raw'].astype(np.float64), a_min=0.0, a_max=None))

q_diff_map, q_ease_map, q_conf_map, default_q_diff, global_q_ease = compute_smoothed_maps(
    train_frame,
    'question_id',
    'response',
    QUESTION_ALPHA,
)
c_diff_map, c_ease_map, c_conf_map, default_c_diff, global_c_ease = compute_smoothed_maps(
    train_frame,
    'concept_id',
    'response',
    CONCEPT_ALPHA,
)

numeric_scaler = StandardScaler()
train_numeric = train_frame[['attempt_log', 'hint_log', 'speed_log']].to_numpy(dtype=np.float32)
train_numeric_scaled = numeric_scaler.fit_transform(train_numeric).astype(np.float32)
train_frame[['attempt_scaled', 'hint_scaled', 'speed_scaled']] = train_numeric_scaled

kmeans = MiniBatchKMeans(
    n_clusters=NUM_BEHAVIOR_PROTOTYPES,
    random_state=SEED,
    batch_size=4096,
    n_init=10,
)
kmeans.fit(train_numeric_scaled)
centers = kmeans.cluster_centers_.astype(np.float32)
question_speed_reference = train_frame.groupby('question_id')['speed_scaled'].median().astype(float).to_dict()
global_speed_reference = float(train_frame['speed_scaled'].median())

for split, split_records in records_by_split.items():
    for uid, record in split_records.items():
        numeric_logs = np.stack([
            np.log1p(np.clip(record['attempts_raw'].astype(np.float64), a_min=0.0, a_max=None)),
            np.log1p(np.clip(record['hints_raw'].astype(np.float64), a_min=0.0, a_max=None)),
            np.log1p(np.clip(record['speed_raw'].astype(np.float64), a_min=0.0, a_max=None)),
        ], axis=-1).astype(np.float32)
        numeric_scaled = numeric_scaler.transform(numeric_logs).astype(np.float32)
        speed = numeric_scaled[:, 2]
        hard_clusters = kmeans.predict(numeric_scaled).astype(np.int32) + 1
        squared_distance = np.sum((numeric_scaled[:, None, :] - centers[None, :, :]) ** 2, axis=-1)
        logits = -squared_distance
        logits = logits - np.max(logits, axis=-1, keepdims=True)
        weights = np.exp(logits)
        weights = weights / np.maximum(np.sum(weights, axis=-1, keepdims=True), 1e-8)
        soft_membership = np.zeros((len(numeric_scaled), NUM_BEHAVIOR_PROTOTYPES + 1), dtype=np.float32)
        soft_membership[:, 1:] = weights.astype(np.float32)
        speed_relative_student = np.zeros(len(speed), dtype=np.float32)
        running_speed_sum = 0.0
        for index, current_speed in enumerate(speed):
            baseline = global_speed_reference if index == 0 else running_speed_sum / index
            speed_relative_student[index] = float(current_speed - baseline)
            running_speed_sum += float(current_speed)
        speed_relative_question = np.asarray(
            [float(current_speed - question_speed_reference.get(int(question_id), global_speed_reference)) for current_speed, question_id in zip(speed, record['question_ids'])],
            dtype=np.float32,
        )
        record['attempts'] = numeric_scaled[:, 0].astype(np.float32)
        record['hints'] = numeric_scaled[:, 1].astype(np.float32)
        record['speed'] = speed.astype(np.float32)
        record['behavior_cluster'] = hard_clusters.astype(np.int32)
        record['behavior_soft_membership'] = soft_membership.astype(np.float32)
        record['speed_relative_student'] = speed_relative_student.astype(np.float32)
        record['speed_relative_question'] = speed_relative_question.astype(np.float32)

train_bundle = build_sequence_bundle(
    records=records_by_split['train'],
    sequence_length=SEQUENCE_LENGTH,
    remainder_min_len=REMAINDER_MIN_LEN,
    q_diff_map=q_diff_map,
    c_diff_map=c_diff_map,
    default_q_diff=default_q_diff,
    default_c_diff=default_c_diff,
    q_ease_map=q_ease_map,
    c_ease_map=c_ease_map,
    q_conf_map=q_conf_map,
    c_conf_map=c_conf_map,
    default_q_ease=global_q_ease,
    default_c_ease=global_c_ease,
    default_confidence=0.0,
    num_behavior_clusters=NUM_BEHAVIOR_PROTOTYPES + 1,
)
valid_bundle = build_sequence_bundle(
    records=records_by_split['valid'],
    sequence_length=SEQUENCE_LENGTH,
    remainder_min_len=REMAINDER_MIN_LEN,
    q_diff_map=q_diff_map,
    c_diff_map=c_diff_map,
    default_q_diff=default_q_diff,
    default_c_diff=default_c_diff,
    q_ease_map=q_ease_map,
    c_ease_map=c_ease_map,
    q_conf_map=q_conf_map,
    c_conf_map=c_conf_map,
    default_q_ease=global_q_ease,
    default_c_ease=global_c_ease,
    default_confidence=0.0,
    num_behavior_clusters=NUM_BEHAVIOR_PROTOTYPES + 1,
)
test_bundle = build_sequence_bundle(
    records=records_by_split['test'],
    sequence_length=SEQUENCE_LENGTH,
    remainder_min_len=REMAINDER_MIN_LEN,
    q_diff_map=q_diff_map,
    c_diff_map=c_diff_map,
    default_q_diff=default_q_diff,
    default_c_diff=default_c_diff,
    q_ease_map=q_ease_map,
    c_ease_map=c_ease_map,
    q_conf_map=q_conf_map,
    c_conf_map=c_conf_map,
    default_q_ease=global_q_ease,
    default_c_ease=global_c_ease,
    default_confidence=0.0,
    num_behavior_clusters=NUM_BEHAVIOR_PROTOTYPES + 1,
)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
save_bundle_npz(train_bundle, TRAIN_NPZ_PATH)
save_bundle_npz(valid_bundle, VALID_NPZ_PATH)
save_bundle_npz(test_bundle, TEST_NPZ_PATH)

metadata = {
    'dataset_name': 'assist2009',
    'source_dir': str(DATA_DIR),
    'source_files': {
        'pickle': str(DATA_DIR / 'assist2009_ahs.pkl'),
        'train_valid_csv': str(DATA_DIR / 'train_valid.csv'),
        'test_csv': str(DATA_DIR / 'test.csv'),
        'keyid2idx': str(DATA_DIR / 'keyid2idx.json'),
    },
    'valid_fold': int(VALID_FOLD),
    'sequence_length': int(SEQUENCE_LENGTH),
    'remainder_min_len': int(REMAINDER_MIN_LEN),
    'num_questions': int(max(question_index.values())),
    'num_concepts': int(max(concept_index.values())),
    'num_question_difficulty': int(max(q_diff_map.values())),
    'num_concept_difficulty': int(max(c_diff_map.values())),
    'num_behavior_clusters': int(NUM_BEHAVIOR_PROTOTYPES + 1),
    'split_summary': {
        'train_users': int(len(records_by_split['train'])),
        'valid_users': int(len(records_by_split['valid'])),
        'test_users': int(len(records_by_split['test'])),
        'train_sequences': int(train_bundle.num_samples),
        'valid_sequences': int(valid_bundle.num_samples),
        'test_sequences': int(test_bundle.num_samples),
    },
    'smoothing': {
        'global_easiness': float(global_q_ease),
        'question_alpha': float(QUESTION_ALPHA),
        'concept_alpha': float(CONCEPT_ALPHA),
    },
    'global_speed_reference': float(global_speed_reference),
}
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

config_payload = {
    'project_name': 'ahs-kt',
    'task_name': TASK_NAME,
    'seed': int(SEED),
    'dataset': {
        'mode': 'real_npz',
        'train_path': str(TRAIN_NPZ_PATH.relative_to(PROJECT_ROOT)),
        'valid_path': str(VALID_NPZ_PATH.relative_to(PROJECT_ROOT)),
        'test_path': str(TEST_NPZ_PATH.relative_to(PROJECT_ROOT)),
    },
    'model': {
        'num_questions': int(metadata['num_questions']),
        'num_concepts': int(metadata['num_concepts']),
        'num_question_difficulty': int(metadata['num_question_difficulty']),
        'num_concept_difficulty': int(metadata['num_concept_difficulty']),
        'num_behavior_clusters': int(metadata['num_behavior_clusters']),
        'sequence_length': int(metadata['sequence_length']),
        'embedding_dim': 64,
        'difficulty_dim': 32,
        'behavior_dim': 32,
        'hidden_dim': 96,
        'dropout': 0.2,
        'use_behavior_cluster': True,
        'use_difficulty_features': True,
        'use_behavior_features': True,
        'use_target_interaction': True,
        'use_relative_speed': True,
        'use_soft_behavior_prototypes': True,
        'question_global_easiness': float(metadata['smoothing']['global_easiness']),
        'concept_global_easiness': float(metadata['smoothing']['global_easiness']),
        'fusion_mode': 'late_residual',
        'behavior_condition_on_difficulty': False,
        'aux_residual_scale': 0.1,
        'difficulty_mode': 'smoothed_target_calibration',
        'difficulty_bias_scale': 0.05,
        'difficulty_feature_source': 'question_only',
    },
    'training': {
        'epochs': int(EPOCHS),
        'batch_size': int(BATCH_SIZE),
        'learning_rate': float(LEARNING_RATE),
        'patience': int(PATIENCE),
    },
    'demo': {
        'train_size': 0,
        'valid_size': 0,
        'test_size': 0,
    },
    'outputs': {
        'root_dir': str(OUTPUT_ROOT.relative_to(PROJECT_ROOT)),
    },
}
CONFIG_PATH.write_text(json.dumps(config_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved metadata to', METADATA_PATH)
print('saved config to', CONFIG_PATH)
print(json.dumps(metadata, ensure_ascii=False, indent=2))

split users = {'train': 2465, 'valid': 617, 'test': 770}
num_questions = 17737
num_concepts = 149
saved metadata to /root/autodl-tmp/ahs-kt/data/assist2009_all_20260421/assist2009_all_20260421_metadata.json
saved config to /root/autodl-tmp/ahs-kt/configs/ahskt_assist2009_all_20260421.json
{
  "dataset_name": "assist2009",
  "source_dir": "/root/autodl-tmp/ahs-kt/data/assist2009",
  "source_files": {
    "pickle": "/root/autodl-tmp/ahs-kt/data/assist2009/assist2009_ahs.pkl",
    "train_valid_csv": "/root/autodl-tmp/ahs-kt/data/assist2009/train_valid.csv",
    "test_csv": "/root/autodl-tmp/ahs-kt/data/assist2009/test.csv",
    "keyid2idx": "/root/autodl-tmp/ahs-kt/data/assist2009/keyid2idx.json"
  },
  "valid_fold": 0,
  "sequence_length": 100,
  "remainder_min_len": 10,
  "num_questions": 17737,
  "num_concepts": 149,
  "num_question_difficulty": 97,
  "num_concept_difficulty": 92,
  "num_behavior_clusters": 5,
  "split_summary": {
    "train_users": 2465,
    "valid_users": 617,
    "t

## 3. 加载 bundle，训练 AHS-KT，并补充 F1

这里调用的是项目原生训练栈：

- `load_config`
- `load_bundle_from_config`
- `AHSKTModel`
- `fit_and_evaluate`

最后会把原生指标和带 `f1` 的指标都落盘。

In [4]:
config = load_config(CONFIG_PATH, project_root=PROJECT_ROOT)
train_bundle, valid_bundle, test_bundle = load_bundle_from_config(config)

print('train / valid / test 序列数 =', train_bundle.num_samples, valid_bundle.num_samples, test_bundle.num_samples)
print('sequence_length =', train_bundle.sequence_length)
print('train bundle shapes:')
for key, value in train_bundle.as_dict().items():
    print(f'  {key}: {value.shape}, dtype={value.dtype}')

tf.keras.backend.clear_session()
model = AHSKTModel(config.model)
train_start = time.time()
metrics_summary = fit_and_evaluate(
    model=model,
    train_bundle=train_bundle,
    valid_bundle=valid_bundle,
    test_bundle=test_bundle,
    config=config,
)
train_elapsed = time.time() - train_start

metrics_path = config.output_root / f'{config.task_name}_metrics.json'
metrics_path.write_text(json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('训练耗时(秒) =', round(train_elapsed, 2))
print('metrics_path =', metrics_path)
print(json.dumps(metrics_summary, ensure_ascii=False, indent=2))

test_targets, test_predictions = collect_targets_and_predictions(
    model=model,
    bundle=test_bundle,
    batch_size=config.training.batch_size,
)
metrics_with_f1 = json.loads(json.dumps(metrics_summary))
metrics_with_f1['test_metrics']['f1'] = float(f1_score(test_targets, (test_predictions >= 0.5).astype(np.int32)))
metrics_with_f1['test_metrics']['auc_recomputed'] = float(roc_auc_score(test_targets, test_predictions))
metrics_with_f1['test_metrics']['acc_recomputed'] = float(accuracy_score(test_targets, (test_predictions >= 0.5).astype(np.int32)))
metrics_with_f1['train_seconds'] = float(train_elapsed)

metrics_with_f1_path = config.output_root / f'{config.task_name}_metrics_with_f1.json'
metrics_with_f1_path.write_text(json.dumps(metrics_with_f1, ensure_ascii=False, indent=2), encoding='utf-8')

print('metrics_with_f1_path =', metrics_with_f1_path)
print(json.dumps(metrics_with_f1, ensure_ascii=False, indent=2))

train / valid / test 序列数 = 3008 716 899
sequence_length = 100
train bundle shapes:
  question_ids: (3008, 100), dtype=int32
  concept_ids: (3008, 100), dtype=int32
  responses: (3008, 100), dtype=int32
  question_difficulty: (3008, 100), dtype=int32
  concept_difficulty: (3008, 100), dtype=int32
  attempts: (3008, 100), dtype=float32
  hints: (3008, 100), dtype=float32
  speed: (3008, 100), dtype=float32
  behavior_cluster: (3008, 100), dtype=int32
  mask: (3008, 100), dtype=int32
  question_easiness: (3008, 100), dtype=float32
  concept_easiness: (3008, 100), dtype=float32
  question_confidence: (3008, 100), dtype=float32
  concept_confidence: (3008, 100), dtype=float32
  speed_relative_student: (3008, 100), dtype=float32
  speed_relative_question: (3008, 100), dtype=float32
  behavior_soft_membership: (3008, 100, 5), dtype=float32
训练耗时(秒) = 37.17
metrics_path = /root/autodl-tmp/ahs-kt/outputs/assist2009_all_20260421_run/ahskt_assist2009_all_20260421_metrics.json
{
  "task_name": "ahs

## 4. Notebook 产物

如果上面的单元全部执行成功，这本 Notebook 会产出：

- `data/assist2009_all_20260421/assist2009_all_20260421_train_ahskt.npz`
- `data/assist2009_all_20260421/assist2009_all_20260421_valid_ahskt.npz`
- `data/assist2009_all_20260421/assist2009_all_20260421_test_ahskt.npz`
- `data/assist2009_all_20260421/assist2009_all_20260421_metadata.json`
- `configs/ahskt_assist2009_all_20260421.json`
- `outputs/assist2009_all_20260421_run/ahskt_assist2009_all_20260421_metrics.json`
- `outputs/assist2009_all_20260421_run/ahskt_assist2009_all_20260421_metrics_with_f1.json`

也就是说，这本 `20260421-assist09-all.ipynb` 已经把“数据目录直连 AHS-KT 并完整跑通”这件事做完整了。